In [ ]:
```xml
<VSCode.Cell language="markdown">
# 🏗️ Microservices Architecture & Service Design Guide

**Designing, building, and orchestrating distributed services for scalability**

This guide covers:
- Service-oriented architecture patterns
- Service discovery and load balancing
- Inter-service communication (sync and async)
- Distributed transactions and saga pattern
- Circuit breakers and resilience
- Container orchestration (Docker, Kubernetes)
- Service mesh (Istio, Linkerd)
- Deployment strategies
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Microservices Patterns

### Service Decomposition
```
Aria Platform - Microservices Architecture

┌─────────────────────────────────────────────────────┐
│                   API Gateway                        │
│         (routing, auth, rate limiting)              │
└─────────────────────────────────────────────────────┘
           │              │              │
     ┌─────▼─────┐  ┌────▼─────┐  ┌────▼─────┐
     │   Chat    │  │ Quantum  │  │ Training │
     │  Service  │  │ Service  │  │ Service  │
     └────┬──────┘  └────┬─────┘  └────┬─────┘
          │              │             │
     ┌────▼──────────────▼─────────────▼────┐
     │         Message Bus (RabbitMQ)       │
     └────┬──────────────┬─────────────┬────┘
          │              │             │
     ┌────▼─────┐  ┌────▼─────┐  ┌───▼──────┐
     │Database  │  │Cache     │  │Storage   │
     │Service   │  │Service   │  │Service   │
     └──────────┘  └──────────┘  └──────────┘
```

### Service Boundaries
```typescript
// Each service owns its data
interface ChatService {
  database: {
    messages: 'chat_db',
    users: 'chat_db',
    sessions: 'chat_db'
  },
  api: {
    '/api/chat/send': 'POST',
    '/api/chat/history': 'GET',
    '/api/chat/status': 'GET'
  }
}

interface TrainingService {
  database: {
    jobs: 'training_db',
    models: 'training_db',
    metrics: 'training_db'
  },
  api: {
    '/api/training/submit': 'POST',
    '/api/training/status': 'GET',
    '/api/models/list': 'GET'
  }
}

// Services coordinate via messages
interface ServiceEvent {
  service: string;
  event_type: string;
  payload: any;
  timestamp: Date;
}
```
</MySCode.Cell>

<VSCode.Cell language="markdown">
## 2. Inter-Service Communication

### Synchronous (REST/gRPC)
```typescript
// REST: Simple, stateless
async function getTrainingJobStatus(jobId: string) {
  const response = await fetch(
    `http://training-service:8001/api/training/jobs/${jobId}`,
    { headers: { 'Authorization': getBearerToken() } }
  );
  return response.json();
}

// gRPC: High-performance, strongly typed
service TrainingService {
  rpc GetJobStatus(JobId) returns (JobStatus);
  rpc SubmitJob(JobRequest) returns (JobResponse);
}

// Usage (via generated client)
const client = new TrainingServiceClient('grpc://training-service:50051');
const status = await client.getJobStatus({ id: jobId });
```

### Asynchronous (Message Queue)
```python
# RabbitMQ: Publish-subscribe
import pika

class MessagePublisher:
    def __init__(self):
        self.connection = pika.BlockingConnection(
            pika.ConnectionParameters('rabbitmq')
        )
        self.channel = self.connection.channel()
    
    def publish_event(self, exchange: str, routing_key: str, message: dict):
        """Publish event to all subscribers"""
        self.channel.exchange_declare(
            exchange=exchange,
            exchange_type='topic',
            durable=True
        )
        
        self.channel.basic_publish(
            exchange=exchange,
            routing_key=routing_key,
            body=json.dumps(message),
            properties=pika.BasicProperties(
                delivery_mode=2,  # Persistent
                content_type='application/json'
            )
        )
        
        logger.info(f"Published event: {routing_key}")

# Publishing from Chat Service
publisher = MessagePublisher()
publisher.publish_event(
    exchange='aria_events',
    routing_key='chat.message_received',
    message={
        'user_id': 'user123',
        'message': 'Hello Aria!',
        'timestamp': datetime.now().isoformat()
    }
)

# Subscribing from Analytics Service
class EventSubscriber:
    def __init__(self):
        self.connection = pika.BlockingConnection(
            pika.ConnectionParameters('rabbitmq')
        )
        self.channel = self.connection.channel()
    
    def subscribe(self, exchange: str, routing_key: str, callback):
        """Subscribe to events"""
        self.channel.exchange_declare(
            exchange=exchange,
            exchange_type='topic'
        )
        
        queue = self.channel.queue_declare(exclusive=True).method.queue
        self.channel.queue_bind(
            exchange=exchange,
            queue=queue,
            routing_key=routing_key
        )
        
        self.channel.basic_consume(
            queue=queue,
            on_message_callback=callback,
            auto_ack=True
        )
        
        self.channel.start_consuming()

# Usage
def process_chat_event(ch, method, properties, body):
    event = json.loads(body)
    logger.info(f"Received: {event}")
    # Log to analytics

subscriber = EventSubscriber()
subscriber.subscribe(
    exchange='aria_events',
    routing_key='chat.#',
    callback=process_chat_event
)
```
</MySCode.Cell>

<VSCode.Cell language="markdown">
## 3. Distributed Transactions

### Saga Pattern (Choreography)
```
Chat Message Processing:
Chat Service → [publish: message.created]
                    ↓
          Analytics Service [received]
                    ↓
          [publish: message.analyzed]
                    ↓
          Storage Service [received]
                    ↓
          [publish: message.stored]
                    ↓
          Cache Service [received]
                    ↓
          [publish: cache.updated]
                    ↓
          Notification Service [received]
                    ↓
          [publish: notification.sent]
```

```python
# Choreography-based saga
class ChatService:
    async def send_message(self, user_id: str, content: str):
        # Step 1: Create message
        message = await self.create_message(user_id, content)
        
        # Step 2: Publish event (triggers other services)
        await self.publish_event('message.created', {
            'message_id': message.id,
            'user_id': user_id,
            'content': content
        })
        
        # Step 3: Subscribe to completion
        await self.wait_for_completion(message.id)
        
        return message

class AnalyticsService:
    async def on_message_created(self, event):
        """Triggered by message.created event"""
        message_id = event['message_id']
        
        # Process message
        analysis = await self.analyze_message(event['content'])
        
        # Publish event
        await publish_event('message.analyzed', {
            'message_id': message_id,
            'sentiment': analysis.sentiment,
            'topics': analysis.topics
        })

# Failure handling (compensating transactions)
class CompensatingTransaction:
    async def handle_failure(self, message_id: str):
        """Rollback on failure"""
        await publish_event('message.failed', {
            'message_id': message_id,
            'action': 'rollback'
        })
```

### Distributed Transactions (with Coordinator)
```python
# Orchestration-based saga with coordinator
class SagaCoordinator:
    async def process_message_transaction(self, user_id: str, content: str):
        """Orchestrate multi-step transaction"""
        
        saga_id = str(uuid4())
        steps = []
        
        try:
            # Step 1: Create message
            message = await self.chat_service.create_message(user_id, content)
            steps.append(('chat.create', message.id))
            
            # Step 2: Analyze
            analysis = await self.analytics_service.analyze(content)
            steps.append(('analytics.analyze', analysis.id))
            
            # Step 3: Store
            storage = await self.storage_service.store(message)
            steps.append(('storage.store', storage.id))
            
            # Step 4: Cache
            await self.cache_service.cache(message)
            steps.append(('cache.update', True))
            
            # Success
            await self.log_saga_completion(saga_id, steps)
            return message
            
        except Exception as e:
            # Compensate in reverse order
            await self.compensate_transaction(saga_id, steps)
            raise
    
    async def compensate_transaction(self, saga_id: str, steps):
        """Rollback in reverse order"""
        for step_name, step_data in reversed(steps):
            if step_name == 'chat.create':
                await self.chat_service.delete_message(step_data)
            elif step_name == 'analytics.analyze':
                await self.analytics_service.delete(step_data)
            elif step_name == 'storage.store':
                await self.storage_service.delete(step_data)
            elif step_name == 'cache.update':
                await self.cache_service.invalidate()
        
        logger.warn(f"Saga {saga_id} compensated")
```
</MySCode.Cell>

<MySCode.Cell language="markdown">
## 4. Resilience Patterns

### Circuit Breaker
```typescript
// Prevent cascading failures
class CircuitBreaker {
  state: 'closed' | 'open' | 'half-open' = 'closed';
  failureCount = 0;
  failureThreshold = 5;
  successThreshold = 2;
  timeout = 60000; // 60 seconds
  
  async call<T>(fn: () => Promise<T>): Promise<T> {
    if (this.state === 'open') {
      if (this.shouldAttemptReset()) {
        this.state = 'half-open';
      } else {
        throw new Error('Circuit breaker is open');
      }
    }
    
    try {
      const result = await fn();
      this.onSuccess();
      return result;
    } catch (error) {
      this.onFailure();
      throw error;
    }
  }
  
  onSuccess() {
    this.failureCount = 0;
    if (this.state === 'half-open') {
      this.state = 'closed';
    }
  }
  
  onFailure() {
    this.failureCount++;
    if (this.failureCount >= this.failureThreshold) {
      this.state = 'open';
      this.openedAt = Date.now();
    }
  }
  
  shouldAttemptReset(): boolean {
    return Date.now() - this.openedAt > this.timeout;
  }
}

// Usage
const trainingServiceBreaker = new CircuitBreaker();

async function submitTrainingJob(jobConfig) {
  return await trainingServiceBreaker.call(() =>
    fetch('/api/training/submit', { body: JSON.stringify(jobConfig) })
  );
}
```

### Retry with Exponential Backoff
```python
import asyncio
import random

async def call_with_retry(
    func,
    max_retries=3,
    base_delay=1,
    max_delay=32,
    jitter=True
):
    """Retry with exponential backoff"""
    for attempt in range(max_retries):
        try:
            return await func()
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            
            # Calculate delay: 2^attempt * base_delay
            delay = min(
                (2 ** attempt) * base_delay,
                max_delay
            )
            
            # Add jitter to prevent thundering herd
            if jitter:
                delay += random.uniform(0, delay * 0.1)
            
            logger.info(f"Attempt {attempt+1} failed, retrying in {delay}s")
            await asyncio.sleep(delay)

# Usage
async def query_training_status(job_id):
    return await call_with_retry(
        lambda: fetch(f'/api/training/{job_id}/status'),
        max_retries=3
    )
```

### Timeout & Bulkhead
```python
import asyncio
from concurrent.futures import ThreadPoolExecutor

class BulkheadPattern:
    def __init__(self, max_concurrent=10):
        self.semaphore = asyncio.Semaphore(max_concurrent)
    
    async def call_with_bulkhead(self, func):
        """Limit concurrent requests"""
        async with self.semaphore:
            return await func()

# Usage
chat_bulkhead = BulkheadPattern(max_concurrent=100)
training_bulkhead = BulkheadPattern(max_concurrent=20)

async def handle_chat_request():
    return await chat_bulkhead.call_with_bulkhead(
        lambda: asyncio.wait_for(
            fetch('/api/chat/send'),
            timeout=30
        )
    )
```
</MySCode.Cell>

<VSCode.Cell language="markdown">
## 5. Service Deployment

### Docker Compose (Local Dev)
```yaml
# docker-compose.yml
version: '3.8'
services:
  api-gateway:
    image: aria/api-gateway:latest
    ports:
      - "8000:8000"
    environment:
      CHAT_SERVICE_URL: http://chat-service:8001
      TRAINING_SERVICE_URL: http://training-service:8002

  chat-service:
    image: aria/chat-service:latest
    ports:
      - "8001:8001"
    environment:
      DATABASE_URL: postgresql://postgres:password@db:5432/chat
      RABBITMQ_URL: amqp://rabbitmq:5672

  training-service:
    image: aria/training-service:latest
    ports:
      - "8002:8002"
    environment:
      DATABASE_URL: postgresql://postgres:password@db:5432/training
      RABBITMQ_URL: amqp://rabbitmq:5672
    deploy:
      resources:
        reservations:
          devices:
            - driver: nvidia
              count: 1
              capabilities: [gpu]

  db:
    image: postgres:14
    environment:
      POSTGRES_PASSWORD: password
    volumes:
      - postgres_data:/var/lib/postgresql/data

  rabbitmq:
    image: rabbitmq:3.11
    ports:
      - "5672:5672"
      - "15672:15672"

volumes:
  postgres_data:
```

### Kubernetes Deployment
```yaml
# k8s/chat-service-deployment.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: chat-service
spec:
  replicas: 3
  selector:
    matchLabels:
      app: chat-service
  template:
    metadata:
      labels:
        app: chat-service
    spec:
      containers:
      - name: chat-service
        image: aria/chat-service:latest
        ports:
        - containerPort: 8001
        env:
        - name: DATABASE_URL
          valueFrom:
            secretKeyRef:
              name: db-credentials
              key: url
        resources:
          requests:
            memory: "512Mi"
            cpu: "250m"
          limits:
            memory: "1Gi"
            cpu: "500m"
        livenessProbe:
          httpGet:
            path: /health
            port: 8001
          initialDelaySeconds: 10
          periodSeconds: 5
        readinessProbe:
          httpGet:
            path: /ready
            port: 8001
          initialDelaySeconds: 5
          periodSeconds: 3

---
apiVersion: v1
kind: Service
metadata:
  name: chat-service
spec:
  selector:
    app: chat-service
  ports:
  - port: 8001
    targetPort: 8001
  type: ClusterIP

---
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: chat-service-hpa
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: chat-service
  minReplicas: 2
  maxReplicas: 10
  metrics:
  - type: Resource
    resource:
      name: cpu
      target:
        type: Utilization
        averageUtilization: 70
  - type: Resource
    resource:
      name: memory
      target:
        type: Utilization
        averageUtilization: 80
```
</MySCode.Cell>

<MySCode.Cell language="markdown">
## 6. Observability

### Distributed Tracing
```python
from opentelemetry import trace
from opentelemetry.exporter.jaeger import JaegerExporter

# Configure Jaeger
jaeger_exporter = JaegerExporter(
    agent_host_name="jaeger",
    agent_port=6831,
)

tracer = trace.get_tracer(__name__)

@app.post("/api/chat/send")
async def send_message(message: str, user_id: str):
    with tracer.start_as_current_span("send_message") as span:
        span.set_attribute("user_id", user_id)
        span.set_attribute("message_length", len(message))
        
        # Step 1: Store message
        with tracer.start_as_current_span("store_message"):
            stored = await db.store_message(user_id, message)
        
        # Step 2: Analyze
        with tracer.start_as_current_span("analyze_message"):
            analysis = await analytics.analyze(message)
        
        # Step 3: Send response
        with tracer.start_as_current_span("send_response"):
            return {"message_id": stored.id, "analysis": analysis}
```

### Service Mesh (Istio)
```yaml
# istio/circuit-breaker.yaml
apiVersion: networking.istio.io/v1beta1
kind: DestinationRule
metadata:
  name: training-service
spec:
  host: training-service
  trafficPolicy:
    connectionPool:
      tcp:
        maxConnections: 100
      http:
        http1MaxPendingRequests: 100
        maxRequestsPerConnection: 2
    outlierDetection:
      consecutive5xxErrors: 5
      interval: 30s
      baseEjectionTime: 30s
```
</MySCode.Cell>

<MySCode.Cell language="markdown">
## 7. Microservices Checklist

- [ ] **Design**
  - [ ] Define service boundaries
  - [ ] Plan data ownership
  - [ ] Design communication patterns
  - [ ] Plan for failures

- [ ] **Implementation**
  - [ ] Build services independently
  - [ ] Implement circuit breakers
  - [ ] Add retry logic
  - [ ] Add timeouts

- [ ] **Deployment**
  - [ ] Containerize services
  - [ ] Set up container registry
  - [ ] Configure orchestration (K8s)
  - [ ] Set up load balancing

- [ ] **Observability**
  - [ ] Add distributed tracing
  - [ ] Implement health checks
  - [ ] Set up metrics collection
  - [ ] Configure alerting

- [ ] **Operations**
  - [ ] Document service APIs
  - [ ] Plan for scaling
  - [ ] Set up CI/CD
  - [ ] Create runbooks

## Success Metrics

- **MTTR** < 5 minutes (via circuit breakers)
- **Availability** > 99.9% (via replicas)
- **Latency** P99 < 500ms (via caching)
- **Scaling**: Handle 3x load in <2 minutes
</MySCode.Cell>
```